# 04c - ML Forecasting (XGBoost/LightGBM)

**Objective**: Forecast using machine learning models

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
print('✓ Libraries imported')

In [ ]:
# Load feature-engineered data
df = pd.read_csv('../data/clean/maize_features_ml.csv', parse_dates=['date'])
print(f'Dataset: {df.shape}')
df.head()

In [ ]:
# Select features
feature_cols = [c for c in df.columns if c not in ['date', 'price', 'observations', 'season']]
X = df[feature_cols]
y = df['price']
print(f'Features: {len(feature_cols)}')
print(feature_cols)

In [ ]:
# Train-test split
train_size = len(df) - 12
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# Train XGBoost
xgb_model = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mape = np.mean(np.abs((y_test - xgb_pred) / y_test)) * 100
print(f'XGBoost - MAE: {xgb_mae:.2f}, RMSE: {xgb_rmse:.2f}, MAPE: {xgb_mape:.2f}%')

In [ ]:
# Train LightGBM
lgbm_model = LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbose=-1)
lgbm_model.fit(X_train, y_train)
lgbm_pred = lgbm_model.predict(X_test)
lgbm_mae = mean_absolute_error(y_test, lgbm_pred)
lgbm_rmse = np.sqrt(mean_squared_error(y_test, lgbm_pred))
lgbm_mape = np.mean(np.abs((y_test - lgbm_pred) / y_test)) * 100
print(f'LightGBM - MAE: {lgbm_mae:.2f}, RMSE: {lgbm_rmse:.2f}, MAPE: {lgbm_mape:.2f}%')

In [ ]:
# Save best model (XGBoost)
import joblib
joblib.dump(xgb_model, '../models/trained/xgboost_maize_model.pkl')
forecast_df = pd.DataFrame({'date': df.iloc[train_size:]['date'], 'forecast': xgb_pred})
forecast_df.to_csv('../models/trained/xgboost_forecast.csv', index=False)
print('✓ Models saved')